In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json
import re


# ==========================================================
# MODEL PATH
# ==========================================================

CHECKPOINT = "./V5D_Final_Merged_Model"


# ==========================================================
# REQUIRED FIELDS
# ==========================================================

REQUIRED_FIELDS = [
    "topic",
    "conversation_goal",
    "conversation_stage",
    "help_needed",
    "relationship_relevance",
    "risk_level",
    "reason"
]


# ==========================================================
# LOAD MODEL
# ==========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("=" * 100)
print("LOADING V5D MODEL")
print("=" * 100)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT,
    trust_remote_code=True,
)

print("Loading merged model...")

model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

print("✅ V5D Model Loaded Successfully")


# ==========================================================
# SYSTEM PROMPT
# ==========================================================

SYSTEM_PROMPT = """You are Compatifi V5D Conversation Understanding Engine.

Your ONLY task is to analyze and understand the current conversation.

You MUST return ONE valid JSON object.

You MUST provide a value for EVERY field.

NEVER output a key without a value.

REQUIRED JSON FORMAT:

{
  "topic": "string",
  "conversation_goal": "string",
  "conversation_stage": "string",
  "help_needed": "string",
  "relationship_relevance": "string",
  "risk_level": "string",
  "reason": "string"
}

FIELD RULES:

topic:
Main subject being discussed.

conversation_goal:
What participants are trying to achieve or resolve.
If unclear, use "unknown".

conversation_stage:
Choose ONE:
opening
exploring
problem_discussion
decision_making
planning
resolution
casual
unknown

help_needed:
Choose ONE:
none
emotional_support
advice
problem_solving
decision_support
communication_help
information
relationship_guidance

relationship_relevance:
Choose ONE:
none
low
medium
high

risk_level:
Choose ONE:
none
low
medium
high

reason:
Short explanation based only on the conversation.

IMPORTANT RULES:

1. EVERY key MUST have a value.
2. Never leave a field empty.
3. Never output only field names.
4. Use "unknown" when conversation_goal is unclear.
5. Use "none" when help or risk is not evident.
6. Do not invent information.
7. Do not give advice.
8. Do not generate a reply.
9. Do not summarize unnecessarily.
10. Do not output markdown.
11. Do not output <think>.
12. Do not add extra keys.
13. Output valid JSON only.
14. Output must start with {.
15. Output must end with }.
"""


# ==========================================================
# TEST CONVERSATIONS
# ==========================================================

test_conversations = [
    {
        "name": "V5D HARD TEST - Multi-Signal Relationship Situation",
        "relationship": "Partner",
        "conversation": """
Partner: You've seemed distant lately. Did I do something wrong?

User: No, it's not just you. I've been stressed about work, but
I also feel like we've been avoiding some things between us.

Partner: Like what?

User: I don't know. We barely spend time together anymore, and when
we do, we're both tired. Sometimes I wonder if we're just getting
used to being disconnected.

Partner: That worries me too. But I don't want us to make decisions
when we're both stressed.

User: I agree. Maybe we should take some time this weekend to actually
talk about what has been bothering us and figure out what we both need.

Partner: Okay. Let's do that. I don't want things to keep getting worse.
"""
    }
]


# ==========================================================
# GENERATE RESPONSE
# ==========================================================

def generate_response(user_prompt):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            repetition_penalty=1.05,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    return response.strip()


# ==========================================================
# EXTRACT JSON
# ==========================================================

def extract_json(text):

    # Remove think blocks if model produces them
    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL
    ).strip()

    # Find JSON object
    match = re.search(
        r"\{.*\}",
        text,
        re.DOTALL
    )

    if not match:
        return None

    json_text = match.group()

    try:
        return json.loads(json_text)

    except json.JSONDecodeError:
        return None


# ==========================================================
# VALIDATE OUTPUT
# ==========================================================

def validate_output(data):

    if not isinstance(data, dict):
        return False, "Output is not valid JSON"

    # Check exact fields
    missing_fields = []

    for field in REQUIRED_FIELDS:

        if field not in data:
            missing_fields.append(field)

        elif not isinstance(data[field], str):
            missing_fields.append(field)

        elif data[field].strip() == "":
            missing_fields.append(field)

    if missing_fields:

        return (
            False,
            f"Missing or empty fields: {missing_fields}"
        )

    # Check extra fields
    extra_fields = [
        key for key in data.keys()
        if key not in REQUIRED_FIELDS
    ]

    if extra_fields:

        return (
            False,
            f"Unexpected fields: {extra_fields}"
        )

    return True, "Valid complete V5D output"


# ==========================================================
# RETRY PROMPT
# ==========================================================

def create_retry_prompt(original_prompt, bad_output):

    return f"""
Your previous output was INVALID.

Previous output:

{bad_output}

You failed because one or more required fields were missing,
empty, or the JSON was invalid.

Analyze the conversation again.

You MUST return ALL seven fields with values.

Required format:

{{
  "topic": "value",
  "conversation_goal": "value or unknown",
  "conversation_stage": "one allowed value",
  "help_needed": "one allowed value",
  "relationship_relevance": "one allowed value",
  "risk_level": "one allowed value",
  "reason": "concise explanation"
}}

Original task:

{original_prompt}

Return ONLY valid JSON.
"""


# ==========================================================
# RUN TESTS
# ==========================================================

for test in test_conversations:

    print("\n")
    print("=" * 100)
    print(test["name"])
    print("=" * 100)

    user_prompt = f"""
Relationship: {test["relationship"]}

Conversation:

{test["conversation"]}

Analyze the current conversation.

Return complete analysis with ALL required fields.
"""

    # ======================================================
    # FIRST ATTEMPT
    # ======================================================

    print("\nATTEMPT 1")
    print("-" * 100)

    response = generate_response(user_prompt)

    print(response)

    data = extract_json(response)

    valid, message = validate_output(data)

    print("\nVALIDATION")
    print("-" * 100)
    print(message)


    # ======================================================
    # RETRY IF INVALID
    # ======================================================

    if not valid:

        print("\n⚠️ INVALID OUTPUT - RETRYING")
        print("-" * 100)

        retry_prompt = create_retry_prompt(
            user_prompt,
            response
        )

        response = generate_response(retry_prompt)

        print("\nATTEMPT 2")
        print("-" * 100)

        print(response)

        data = extract_json(response)

        valid, message = validate_output(data)

        print("\nFINAL VALIDATION")
        print("-" * 100)

        print(message)


    # ======================================================
    # FINAL RESULT
    # ======================================================

    if valid:

        print("\n" + "=" * 100)
        print("✅ FINAL STRUCTURED OUTPUT")
        print("=" * 100)

        print(
            json.dumps(
                data,
                indent=4,
                ensure_ascii=False
            )
        )

    else:

        print("\n" + "=" * 100)
        print("❌ MODEL FAILED STRUCTURED OUTPUT")
        print("=" * 100)


print("\n" + "=" * 100)
print("✅ V5D TESTING COMPLETE")
print("=" * 100)

g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


LOADING V5D MODEL
Loading tokenizer...
Loading merged model...


Loading checkpoint shards: 100%|██████████| 5/5 [01:10<00:00, 14.11s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ V5D Model Loaded Successfully


V5D HARD TEST - Multi-Signal Relationship Situation

ATTEMPT 1
----------------------------------------------------------------------------------------------------


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<think>

</think>

{"topic":"Relationship","conversation_goal":"Address relationship distance and emotional disconnection caused by stress.","help_needed":"Seek guidance on improving communication and reconnecting with a partner.","relationship_relevance":"Partner","risk_level":"Medium","reason":"The partners are discussing feelings of emotional distance due to work stress and lack of quality time."}

VALIDATION
----------------------------------------------------------------------------------------------------
Missing or empty fields: ['conversation_stage']

⚠️ INVALID OUTPUT - RETRYING
----------------------------------------------------------------------------------------------------

ATTEMPT 2
----------------------------------------------------------------------------------------------------
<think>

{"topic":"Relationship","conversation_goal":"Address emotional distance and lack of communication with a partner.","help_needed":"Seek guidance on resolving relationship tension and i